In [1]:
# Python Standard Library
import sys
from os import mkdir
from os.path import join
from os.path import isdir
from shutil import rmtree

# Community Modules
from tqdm import tqdm
import numpy as np
import pandas as pd
from scipy import stats

# My Modules
sys.path.insert(0, "../code")
import measure_signal as ms
import dataset_utils as du
from matplotlib import pyplot as plt

rng = np.random.default_rng(1415)

2026-03-23 14:39:47.100639: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-23 14:39:47.119823: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-23 14:39:47.125617: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-23 14:39:47.140251: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-23 14:39:49.697450: W tensorflow/compiler/tf2

In [2]:
wvl, df_meta, spectra, spectra_signal, spectra_noise = du.load_dataset()

num_wvl = wvl.size
num_spectra = spectra.shape[0]

In [3]:
new_SNR = 25
new_N = (df_meta["S (SNR)"] / new_SNR).to_numpy()
new_N_arr = np.full((num_spectra, num_wvl), new_N[..., np.newaxis])
new_noise = stats.norm.rvs(loc=0, scale=new_N_arr, random_state=rng)
new_spectra = spectra_signal + new_noise

In [4]:
def plot_with_SNR_stuff(ax, specsnr):
    ax.set_xlim((4500, 7000))
    ax.axis("off")
    
    ax.plot(wvl, specsnr.spectrum, c="k", lw=1)
    ax.plot(wvl, specsnr.signal, c="tab:blue")

    vertical_midpoint = np.sum(ax.get_ylim()) / 2
    horizontal_text_location = ax.get_xlim()[0]
    ax.text(
        horizontal_text_location, vertical_midpoint,
        f"{specsnr.name}\n{specsnr.subtype}\n{specsnr.phase} days",
        ha="right", va="center")

    horizontal_text_location = ax.get_xlim()[1]
    ax.text(
        horizontal_text_location, vertical_midpoint,
        f"SNR = {specsnr.SNR:.2f}\nS = {specsnr.S:.2e}\nN = {specsnr.N:.2e}\n$\sigma = {specsnr.denoising_parameter}$",
        ha="left", va="center")

    ax.plot(specsnr.pc_wvl, specsnr.pseudo_cont, c="tab:green")

    # Color in the regions used to calculate the noise on the red and blue
    # shoulders.
    if specsnr.useBlu:
        ax.fill_between(
            specsnr.wvl[specsnr.blu_inds],
            y1=1,
            y2=0,
            color="tab:blue",
            alpha=0.25)
        ax.fill_between(
            specsnr.wvl[specsnr.blu_inds],
            y1=1,
            y2=0,
            color="tab:blue",
            alpha=0.25)

    if specsnr.useRed:
        ax.fill_between(
            specsnr.wvl[specsnr.red_inds],
            y1=1,
            y2=0,
            color="tab:red",
            alpha=0.25)
        ax.fill_between(
            specsnr.wvl[specsnr.red_inds],
            y1=1,
            y2=0,
            color="tab:red",
            alpha=0.25)
    return


def plot_without_SNR_stuff(ax, specsnr):
    ax.set_xlim((4500, 7000))
    ax.axis("off")
    
    ax.plot(wvl, specsnr.spectrum, c="k", lw=1)
    # ax.plot(wvl, specsnr.signal, c="tab:blue")

    vertical_midpoint = np.sum(ax.get_ylim()) / 2
    horizontal_text_location = ax.get_xlim()[0]
    ax.text(
        horizontal_text_location, vertical_midpoint,
        f"{specsnr.name}\n{specsnr.subtype}\n{specsnr.phase} days",
        ha="right", va="center")

    # horizontal_text_location = ax.get_xlim()[1]
    # ax.text(
    #     horizontal_text_location, vertical_midpoint,
    #     f"SNR = {specsnr.SNR:.2f}\nS = {specsnr.S:.2e}\nN = {specsnr.N:.2e}\n$\sigma = {specsnr.denoising_parameter}$",
    #     ha="left", va="center")

    # ax.plot(specsnr.pc_wvl, specsnr.pseudo_cont, c="tab:green")

    # # Color in the regions used to calculate the noise on the red and blue
    # # shoulders.
    # if specsnr.useBlu:
    #     ax.fill_between(
    #         specsnr.wvl[specsnr.blu_inds],
    #         y1=1,
    #         y2=0,
    #         color="tab:blue",
    #         alpha=0.25)
    #     ax.fill_between(
    #         specsnr.wvl[specsnr.blu_inds],
    #         y1=1,
    #         y2=0,
    #         color="tab:blue",
    #         alpha=0.25)

    # if specsnr.useRed:
    #     ax.fill_between(
    #         specsnr.wvl[specsnr.red_inds],
    #         y1=1,
    #         y2=0,
    #         color="tab:red",
    #         alpha=0.25)
    #     ax.fill_between(
    #         specsnr.wvl[specsnr.red_inds],
    #         y1=1,
    #         y2=0,
    #         color="tab:red",
    #         alpha=0.25)
    return

In [5]:
subtypes_ID = df_meta["SN Subtype ID"].unique()
subtypes_ID_to_str = dict(
    df_meta.groupby(by="SN Subtype ID")["SN Subtype"].apply(
        lambda x: x.to_numpy()[0]))

In [6]:
DIR_FIGS = "../sparklines_redux_SNR25_test"
if isdir(DIR_FIGS):
    rmtree(DIR_FIGS)
mkdir(DIR_FIGS)

for subtype_ID, subtype_str in subtypes_ID_to_str.items():
    dir_subtype_figs = join(DIR_FIGS, f"{subtype_ID:0>2}_{subtype_str}")
    if isdir(dir_subtype_figs):
        rmtree(dir_subtype_figs)
    mkdir(dir_subtype_figs)

In [9]:
for subtype_ID, subtype_str in subtypes_ID_to_str.items():
    num_sparklines_per_fig = 10
    
    dir_subtype_figs = join(DIR_FIGS, f"{subtype_ID:0>2}_{subtype_str}")
    
    mask = (df_meta["SN Subtype ID"] == subtype_ID)
    df_subtype = df_meta[mask].copy(deep=True).reset_index(drop=True)
    subtype_spectra = spectra[mask]
    subtype_new_noise = new_noise[mask]
    # subtype_spectra = new_spectra[mask]
    
    assert mask.sum() == df_subtype.shape[0]
    assert mask.sum() == subtype_spectra.shape[0]
    subtype_num_spectra = mask.sum()
    print(subtype_ID, subtype_str, subtype_num_spectra)
    
    for i in tqdm(range(0, subtype_num_spectra, num_sparklines_per_fig)):
        if not (i + num_sparklines_per_fig <= subtype_num_spectra):
            num_sparklines_per_fig = subtype_num_spectra - i
        
        fig, axes = plt.subplots(
            ncols=1,
            nrows=num_sparklines_per_fig,
            figsize=(6, num_sparklines_per_fig))

        for j in range(num_sparklines_per_fig):
            if num_sparklines_per_fig == 1:
                ax = axes
            else:
                ax = axes[j]
            
            subtype_df_meta_ij = df_subtype.loc[i+j]
            spectrum_ij = subtype_spectra[i+j]
            new_noise_ij = subtype_new_noise[i+j]
            
            assert subtype_df_meta_ij.ndim == 1
            assert spectrum_ij.ndim == 1
            
            specsnr = ms.SpectrumSNR(
                subtype_df_meta_ij["SN Name"],
                subtype_df_meta_ij["SN Subtype"],
                subtype_df_meta_ij["Spectrum Phase"],
                wvl,
                spectrum_ij,
            )
            # try:
            specsnr.execute_algorithm(subtype_df_meta_ij, new_noise=new_noise_ij)
            specsnr.spectrum += new_noise_ij
            SNobjs_before_after[subtype_str].append(specsnr)
            plot_with_SNR_stuff(ax, specsnr)
            # except IndexError:
            #     plot_without_SNR_stuff(ax, specsnr)
        
        fig.tight_layout()
        fig.savefig(join(dir_subtype_figs, f"{i:0>5}"))
        # display(plt.gcf())
        plt.close()

0 Ia-norm 2034


100%|██████████| 204/204 [00:00<00:00, 3025.10it/s]


1 Ia-91T 339


100%|██████████| 34/34 [00:00<00:00, 499.88it/s]


2 Ia-91bg 222


...

100%|██████████| 18/18 [00:00<00:00, 524288.00it/s]


8 Ic-broad 210


100%|██████████| 21/21 [00:00<00:00, 599186.29it/s]


9 IIP 100


100%|██████████| 10/10 [00:00<00:00, 218453.33it/s]
